In [28]:
from rdflib import Graph

g = Graph()
g.parse("data/dbpedia_2015-10.nt", format="nt")

ontology_terms = dict()

for uri, uri_type, label, comment in g.query("""
        SELECT ?s ?type ?label ?comment WHERE { ?s a ?type ; rdfs:label ?label . OPTIONAL { ?s rdfs:comment ?comment . FILTER (lang(?comment) = "en") } . VALUES ?type { owl:Class owl:ObjectProperty owl:DatatypeProperty } . FILTER (lang(?label) = "en") }
    """):
    text = f"{uri}\n\n{label}\n\n{comment}"
    uri_type = uri_type.split('#')[-1]
    ontology_terms[str(uri)] = {
        "uri": str(uri),
        "type": uri_type,
        "label": label.value,
        "comment": comment.value if comment else None,
        "text": text,
    }

len(ontology_terms)
    

3571

In [ ]:
from openai import OpenAI
import dotenv
import os

dotenv.load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")

# response = client.embeddings.create(
#     input="Your text string goes here",
#     model="text-embedding-3-small"
# )

# print(response.data[0].embedding)

In [19]:
def embed_texts(texts):
    embeddings = client.embeddings.create(
        input=[f"Term: {text}" for text in texts],
        model="text-embedding-3-small"
    )
    return [x.embedding for x in embeddings.data]

text_embeddings = []

texts = [x['text'] for x in ontology_terms.values()]

# send in batches of N
N = 1024
for i in range(0, len(texts), N):
    text_embeddings.extend(embed_texts(texts[i : i + N]))
    print(len(text_embeddings))

1024
2048
3072
3572


In [20]:
with open("data/ontology-vectors.w2v", 'w') as f:
    f.write(f"{len(ontology_terms)} {len(text_embeddings[0])}\n")
    for (uri, info), emb in zip(ontology_terms.items(), text_embeddings):
        f.write(f"{info['type']}|{uri} {' '.join([str(x) for x in emb])}\n")

In [21]:
from gensim.models import KeyedVectors

model = KeyedVectors.load_word2vec_format("data/ontology-vectors.w2v")

In [32]:
import numpy as np


def lookup_term(term, classes=True, properties=True, k=5):

    term_emb = embed_texts([f"Term: {term}"])[0]

    results_retrieved = 0

    print(f"Results:")
    for entry, score in model.most_similar(positive=[np.array(term_emb)], topn=100):
        uri_type, uri = entry.split("|")
        if uri_type == "Class" and not classes:
            continue
        if "Property" in uri_type and not properties:
            continue
        uri_info = ontology_terms[uri]
        print(f"- URI: {uri}\n  Score: {score:.2f}\n  Type: {uri_type}\n  Label: {uri_info['label']}\n  Description: {uri_info['comment'] if uri_info['comment'] else 'N/A'}")
        results_retrieved += 1
        if results_retrieved >= k:
            break

term = "author"
lookup_term(term)

Results:
- URI: http://dbpedia.org/ontology/author
  Score: 0.60
  Type: ObjectProperty
  Label: author
  Description: N/A
- URI: http://dbpedia.org/ontology/writer
  Score: 0.50
  Type: ObjectProperty
  Label: writer
  Description: N/A
- URI: http://dbpedia.org/ontology/SongWriter
  Score: 0.46
  Type: Class
  Label: songwriter
  Description: a person who writes songs.
- URI: http://dbpedia.org/ontology/editorTitle
  Score: 0.45
  Type: DatatypeProperty
  Label: editor title
  Description: N/A
- URI: http://dbpedia.org/ontology/creator
  Score: 0.44
  Type: ObjectProperty
  Label: creator (agent)
  Description: Creator/author of a work. For literal (string) use dc:creator; for object (URL) use creator


In [36]:
lookup_term("population total")

Results:
- URI: http://dbpedia.org/ontology/totalPopulation
  Score: 0.67
  Type: DatatypeProperty
  Label: total population
  Description: N/A
- URI: http://dbpedia.org/ontology/previousPopulationTotal
  Score: 0.65
  Type: DatatypeProperty
  Label: previous population total
  Description: N/A
- URI: http://dbpedia.org/ontology/populationTotal
  Score: 0.64
  Type: DatatypeProperty
  Label: population total
  Description: N/A
- URI: http://dbpedia.org/ontology/populationTotalReference
  Score: 0.61
  Type: ObjectProperty
  Label: total population reference
  Description: N/A
- URI: http://dbpedia.org/ontology/population
  Score: 0.60
  Type: ObjectProperty
  Label: population
  Description: N/A
